# Entrenamiento completo y validación en T4

Se continúa el entrenamiento de ambos rangos durante una época adicional completa. Se conservan los adapters iniciales y se selecciona la mayor EX medida con el mismo entorno y prompt. La ejecución requiere una GPU T4.

In [ ]:
from pathlib import Path
import io, json, os, subprocess, sys, zipfile
from google.colab import files
raiz = Path('/content/Deep-Learning')
if not (raiz / 'solucion.py').exists():
    subidos = files.upload()
    archivos = [n for n in subidos if n.endswith('.zip')]
    if len(archivos) != 1:
        raise ValueError('Se requiere un archivo de transferencia del proyecto')
    raiz.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(subidos[archivos[0]])) as z:
        for nombre in z.namelist():
            if not (raiz / nombre).resolve().is_relative_to(raiz.resolve()):
                raise ValueError('Ruta fuera del directorio del proyecto')
        z.extractall(raiz)
os.chdir(raiz)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ.pop('MODEL_LOCAL_PATH', None)
print(raiz)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('La comprobación requiere una GPU T4')
print(torch.__version__, torch.cuda.get_device_name(0), torch.version.cuda)


In [ ]:
preparacion = Path('scripts/colab_mysql.sh').read_text(encoding='utf-8')
subprocess.run(['bash'], input=preparacion, text=True, check=True)
os.environ['MYSQL_SOCKET'] = '/tmp/my8run/mysql.sock'
subprocess.run([sys.executable, '-m', 'generador.construir_bases'], check=True)


In [ ]:
from collections import defaultdict
import hashlib
from generador.bd import conectar
con = conectar()
cur = con.cursor()
cur.execute('SELECT VERSION(), @@sql_mode')
version, modo = cur.fetchone()
con.close()
assert version.startswith('8.4.10') and 'ONLY_FULL_GROUP_BY' in modo
por_esquema = defaultdict(list)
for linea in Path('datos/entrenamiento.jsonl').read_text(encoding='utf-8').splitlines():
    fila = json.loads(linea)
    por_esquema[fila['esquema_id']].append(fila)
aceptados, errores = 0, []
for esquema, filas in por_esquema.items():
    con = conectar('p3_' + esquema)
    try:
        cur = con.cursor()
        cur.execute('SET SESSION TRANSACTION READ ONLY')
        cur.execute('SET SESSION MAX_EXECUTION_TIME=10000')
        for fila in filas:
            try:
                cur.execute(fila['sql'])
                cantidad = len(cur.fetchall())
                if not 1 <= cantidad <= 500:
                    raise ValueError('Cantidad de filas fuera del intervalo admitido')
                aceptados += 1
            except Exception as error:
                errores.append(dict(id=fila['id'], error=str(error)))
    finally:
        con.close()
reporte = dict(motor=version, sql_mode=modo, dispositivo=torch.cuda.get_device_name(0),
               aceptados=aceptados, errores=errores,
               sha256_dataset=hashlib.sha256(Path('datos/entrenamiento.jsonl').read_bytes()).hexdigest())
Path('resultados/validacion_mysql_t4.json').write_text(json.dumps(reporte,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(reporte,ensure_ascii=False,indent=2))
assert aceptados >= 2000 and not errores


In [ ]:
subprocess.run([sys.executable, '-u', 'ejecutar_proyecto.py', '--omitir-datos', '--reanudar', '--ampliar-entrenamiento'], check=True)


In [ ]:
subprocess.run([sys.executable, 'verificar_entrega.py'], check=True)
verificacion = json.loads(Path('resultados/verificacion.json').read_text(encoding='utf-8'))
contrato = json.loads(Path('resultados/contrato_inferencia.json').read_text(encoding='utf-8'))
assert verificacion['completo'] and verificacion['entorno_objetivo_verificado']
assert contrato['cumple'] and 'T4' in contrato['dispositivo']
print(Path('resultados/analisis.md').read_text(encoding='utf-8'))


In [ ]:
from verificar_entrega import archivos_proyecto
salida = Path('/content/resultado_t4.zip')
with zipfile.ZipFile(salida,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in archivos_proyecto():
        z.write(p,p.relative_to(raiz))
files.download(str(salida))
